In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [0]:
# TF-IDF (Term Frequency – Inverse Document Frequency)

In [0]:
books_df = spark.table('workspace.default.dim_books').toPandas()

In [0]:
#turn each book's genres list into plain text
books_df["genre_text"] = books_df["genres"].apply(lambda genre_list: " ".join(genre_list))

In [0]:
#turn text in number, TF-IDF vectors
vectorizer = TfidfVectorizer()
genre_vectors = vectorizer.fit_transform(books_df["genre_text"])

In [0]:
#compare every book to every other book
similarity_scores = cosine_similarity(genre_vectors)

In [0]:
#if it's given a book title, find most similar books
def recommend_similar_books(title, how_many=5):
    #find which row number this book is
    book_index = books_df[books_df["title"] == title].index[0]

    #get that book's similarity score against every other book
    scores_for_this_book = similarity_scores[book_index]

    #pair each score with its row number, sorted highest first
    ranked = sorted(enumerate(scores_for_this_book), key=lambda pair: pair[1], reverse=True)

    #skip the first result
    top_matches = ranked[1:how_many + 1]

    matching_row_numbers = [row_number for row_number, score in top_matches]
    return books_df.iloc[matching_row_numbers][["title", "genres"]]

In [0]:
#testing
print(recommend_similar_books("The Hobbit"))

                                                  title                                     genres
1392                              Babe: The Gallant Pig  [fiction, classics, fantasy, young-adult]
1481                  The Marvelous Land of Oz (Oz, #2)  [fantasy, classics, fiction, young-adult]
1484   The Faraway Tree Stories (The Faraway Tree #1-3)  [fantasy, fiction, classics, young-adult]
1530  The Country Mouse and the City Mouse; The Fox ...  [fiction, classics, fantasy, young-adult]
1639  The Folk of the Faraway Tree (The Faraway Tree...  [fantasy, classics, fiction, young-adult]


In [0]:
#error: index object is not callable, index(0) -> index[0]